# Cross-check: Anthropometric Data Leakage in Stunting Prediction (SKI 2023)

Notebook ini menjalankan ulang **seluruh** eksperimen naskah IJECE *"Anthropometric data leakage inflates machine learning performance in child stunting prediction"* dan **membandingkan otomatis** setiap angka dengan hasil yang dilaporkan di naskah.

**Yang dibutuhkan (2 file):**
1. `SKI_2023_Balita_0-59_Bulan_asli_labels.xlsx` — data SKI 2023 balita 0–59 bulan
2. `lenanthro.txt` — tabel WHO length/height-for-age (LMS harian)

**Urutan:** jalankan sel dari atas ke bawah (`Runtime → Run all`). Estimasi waktu di Colab gratis: baca Excel ±3–5 menit, pelatihan 12 model ±8–12 menit, CV & permutation importance (opsional) ±15 menit.

Status cross-check: ✅ = cocok dengan naskah (dalam toleransi), ⚠️ = berbeda (lihat catatan di bagian akhir).

## 1. Instalasi versi pustaka
Untuk mereproduksi angka **persis**, gunakan versi yang sama dengan analisis asli. Setelah sel ini selesai, Colab mungkin meminta **Restart session** — lakukan, lalu lanjutkan dari sel berikutnya (tidak perlu menjalankan sel ini lagi).

In [ ]:
!pip -q install "scikit-learn==1.8.0" "openpyxl>=3.1"

## 2. Impor & pengaturan

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, sklearn, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, f1_score, precision_score,
                             recall_score, balanced_accuracy_score, matthews_corrcoef, brier_score_loss,
                             confusion_matrix, roc_curve)
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
SEED = 42
RUN_CV = True        # validasi silang 5-fold grouped (±5-8 menit)
RUN_PERM = True      # permutation importance (±6-8 menit)
N_BOOT = 1000
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
if sklearn.__version__ != "1.8.0":
    print("⚠️ Versi scikit-learn berbeda dari analisis asli (1.8.0); angka bisa bergeser sedikit di digit ke-3.")

## 3. Lokasi data
Pilih **salah satu**: (A) Google Drive — ubah `DATA_DIR` sesuai folder Anda; atau (B) unggah langsung (set `USE_UPLOAD = True`).

In [ ]:
USE_UPLOAD = False
DATA_DIR = "/content/drive/MyDrive/Anthropometric_Data_Leakage/data"   # <- sesuaikan
DATA_FILE = "SKI_2023_Balita_0-59_Bulan_asli_labels.xlsx"
WHO_FILE = "lenanthro.txt"

try:
    from google.colab import drive, files
    if USE_UPLOAD:
        up = files.upload(); DATA_DIR = "/content"
    else:
        drive.mount("/content/drive")
except ImportError:
    pass  # bukan di Colab: gunakan DATA_DIR lokal
DATA_PATH = os.path.join(DATA_DIR, DATA_FILE); WHO_PATH = os.path.join(DATA_DIR, WHO_FILE)
print(DATA_PATH, os.path.exists(DATA_PATH)); print(WHO_PATH, os.path.exists(WHO_PATH))

## 4. Nilai acuan dari naskah (untuk cross-check)
Angka di bawah diambil dari `results/*.csv` run final yang dipakai di naskah.

In [ ]:
EXPECTED = {
 "metrics": {
  "Baseline|Majority/prior": {
   "AUC-ROC": 0.5,
   "PR-AUC": 0.2354,
   "Accuracy": 0.7646,
   "Recall": 0.0,
   "Specificity": 1.0,
   "F1": 0.0,
   "MCC": 0.0
  },
  "B (determinants only)|Logistic Regression": {
   "AUC-ROC": 0.656,
   "PR-AUC": 0.3564,
   "Accuracy": 0.6103,
   "Recall": 0.6146,
   "Specificity": 0.609,
   "F1": 0.4262,
   "MCC": 0.1909
  },
  "B (determinants only)|Random Forest": {
   "AUC-ROC": 0.6537,
   "PR-AUC": 0.3579,
   "Accuracy": 0.7444,
   "Recall": 0.2131,
   "Specificity": 0.908,
   "F1": 0.2819,
   "MCC": 0.1579
  },
  "B (determinants only)|Gradient Boosting (HGB)": {
   "AUC-ROC": 0.6703,
   "PR-AUC": 0.3729,
   "Accuracy": 0.6293,
   "Recall": 0.6129,
   "Specificity": 0.6344,
   "F1": 0.4377,
   "MCC": 0.2123
  },
  "A1 (B + weight)|Logistic Regression": {
   "AUC-ROC": 0.8027,
   "PR-AUC": 0.5581,
   "Accuracy": 0.721,
   "Recall": 0.7604,
   "Specificity": 0.7088,
   "F1": 0.562,
   "MCC": 0.4061
  },
  "A1 (B + weight)|Random Forest": {
   "AUC-ROC": 0.7586,
   "PR-AUC": 0.4831,
   "Accuracy": 0.7714,
   "Recall": 0.377,
   "Specificity": 0.8928,
   "F1": 0.4371,
   "MCC": 0.3043
  },
  "A1 (B + weight)|Gradient Boosting (HGB)": {
   "AUC-ROC": 0.8251,
   "PR-AUC": 0.6198,
   "Accuracy": 0.7633,
   "Recall": 0.7344,
   "Specificity": 0.7722,
   "F1": 0.5937,
   "MCC": 0.4515
  },
  "A2 (B + all anthropometry)|Logistic Regression": {
   "AUC-ROC": 0.951,
   "PR-AUC": 0.8194,
   "Accuracy": 0.8821,
   "Recall": 0.9079,
   "Specificity": 0.8742,
   "F1": 0.7839,
   "MCC": 0.7175
  },
  "A2 (B + all anthropometry)|Random Forest": {
   "AUC-ROC": 0.8624,
   "PR-AUC": 0.6471,
   "Accuracy": 0.8184,
   "Recall": 0.5359,
   "Specificity": 0.9053,
   "F1": 0.5815,
   "MCC": 0.4693
  },
  "A2 (B + all anthropometry)|Gradient Boosting (HGB)": {
   "AUC-ROC": 0.9983,
   "PR-AUC": 0.9949,
   "Accuracy": 0.9778,
   "Recall": 0.9796,
   "Specificity": 0.9772,
   "F1": 0.9541,
   "MCC": 0.94
  },
  "L0 (height + age + sex)|Logistic Regression": {
   "AUC-ROC": 0.9097,
   "PR-AUC": 0.6794,
   "Accuracy": 0.8299,
   "Recall": 0.8371,
   "Specificity": 0.8277,
   "F1": 0.6985,
   "MCC": 0.6003
  },
  "L0 (height + age + sex)|Random Forest": {
   "AUC-ROC": 0.9989,
   "PR-AUC": 0.9967,
   "Accuracy": 0.9838,
   "Recall": 0.9818,
   "Specificity": 0.9845,
   "F1": 0.9662,
   "MCC": 0.9558
  },
  "L0 (height + age + sex)|Gradient Boosting (HGB)": {
   "AUC-ROC": 0.9986,
   "PR-AUC": 0.9958,
   "Accuracy": 0.9782,
   "Recall": 0.9831,
   "Specificity": 0.9767,
   "F1": 0.955,
   "MCC": 0.9413
  }
 },
 "delta": {
  "Logistic Regression|A1 (B + weight) vs B": 0.1467,
  "Logistic Regression|A2 (B + all anthropometry) vs B": 0.295,
  "Logistic Regression|L0 (height + age + sex) vs B": 0.2537,
  "Random Forest|A1 (B + weight) vs B": 0.1048,
  "Random Forest|A2 (B + all anthropometry) vs B": 0.2086,
  "Random Forest|L0 (height + age + sex) vs B": 0.3452,
  "Gradient Boosting (HGB)|A1 (B + weight) vs B": 0.1549,
  "Gradient Boosting (HGB)|A2 (B + all anthropometry) vs B": 0.3281,
  "Gradient Boosting (HGB)|L0 (height + age + sex) vs B": 0.3284
 },
 "cv": {
  "A1 (B + weight)": 0.8218,
  "A2 (B + all anthropometry)": 0.9984,
  "B (determinants only)": 0.6705,
  "L0 (height + age + sex)": 0.9986
 },
 "flow": {
  "raw": 86364,
  "valid_age_0_59m": 86363,
  "height_available": 85169,
  "plausible_HAZ_abs_le_6": 84205,
  "stunted_n": 19825,
  "prevalence_unweighted": 0.2354,
  "prevalence_weighted": 0.2198,
  "train_n": 67364,
  "test_n": 16841,
  "train_prev": 0.2354,
  "test_prev": 0.2354,
  "households_train": 60286,
  "households_test": 15060
 }
}
TOL_COUNT = 0          # jumlah sampel harus persis sama
TOL_PREV = 0.001       # prevalensi
TOL_AUC = 0.005        # AUC / metrik model (toleransi perbedaan versi pustaka)
checks = []
def check(name, got, exp, tol):
    ok = abs(got - exp) <= tol
    checks.append({"check": name, "notebook": round(float(got), 4), "manuscript": exp, "abs_diff": round(abs(got-exp), 4), "status": "✅" if ok else "⚠️"})
    print(f"{'✅' if ok else '⚠️'} {name}: notebook={got:.4f} | naskah={exp} ")

## 5. Baca data

In [ ]:
t0 = time.time()
df = pd.read_pickle(DATA_PATH) if DATA_PATH.endswith(".pkl") else pd.read_excel(DATA_PATH, engine="openpyxl")
print(df.shape, f"({time.time()-t0:.0f} detik)")

## 6. Variabel target: HAZ menurut WHO Child Growth Standards
$HAZ = \\frac{(y/M)^L - 1}{L \\cdot S}$ dengan L, M, S dari `lenanthro.txt` per jenis kelamin dan umur (hari). Koreksi posisi ukur WHO: < 731 hari diukur berdiri → +0,7 cm; ≥ 731 hari diukur telentang → −0,7 cm. Stunting = HAZ < −2.

In [ ]:
who = pd.read_csv(WHO_PATH, sep=r"\s+"); who.columns = [c.lower() for c in who.columns]
REF = {(int(r.sex), int(r.age)): (r.l, r.m, r.s) for r in who.itertuples()}

def parse_date(x):
    s = str(int(x)).zfill(8)
    return pd.Timestamp(year=int(s[4:]), month=int(s[2:4]), day=int(s[:2]))
birth = df["6. Tanggal Lahir"].map(lambda x: parse_date(x) if pd.notna(x) else pd.NaT)
visit = df["2. Tanggal Pengumpulan data: (tgl-bln)"].map(lambda x: parse_date(x) if pd.notna(x) else pd.NaT)
age_days = (visit - birth).dt.days.astype(float).values
sex = np.where(df["4. Jenis Kelamin"].eq("Laki-laki"), 1, 2)
POS = "J02.c.KHUSUS UNTUK BALITA, (Posisi pengukuran TB/PB)"; HGT = "J02.b.Tinggi/Panjang Badan (cm)"
recumbent = df[POS].eq("Telentang").values
h = df[HGT].astype(float).values.copy()
h = np.where((age_days < 731) & ~recumbent, h + 0.7, h)
h = np.where((age_days >= 731) & recumbent, h - 0.7, h)
haz = np.full(len(df), np.nan)
for i in range(len(df)):
    if np.isnan(h[i]) or np.isnan(age_days[i]): continue
    key = (sex[i], int(round(age_days[i])))
    if key in REF:
        L, M, S = REF[key]; haz[i] = ((h[i] / M) ** L - 1) / (L * S)
df["age_days"], df["HAZ"] = age_days, haz

flow = {"raw": len(df)}
m = df["age_days"].between(0, 1856); flow["valid_age_0_59m"] = int(m.sum())
m &= df[HGT].notna() & df[POS].notna(); flow["height_available"] = int(m.sum())
m &= df["HAZ"].between(-6, 6); flow["plausible_HAZ_abs_le_6"] = int(m.sum())
df = df[m].reset_index(drop=True)
y = (df["HAZ"] < -2).astype(int).values
w = df["Penimbang Populasi Individu"].values
flow["stunted_n"] = int(y.sum()); flow["prevalence_unweighted"] = y.mean(); flow["prevalence_weighted"] = np.average(y, weights=w)
for k in ["raw", "valid_age_0_59m", "height_available", "plausible_HAZ_abs_le_6", "stunted_n"]:
    check(f"Alur data: {k}", flow[k], EXPECTED["flow"][k], TOL_COUNT)
for k in ["prevalence_unweighted", "prevalence_weighted"]:
    check(f"Alur data: {k}", flow[k], EXPECTED["flow"][k], TOL_PREV)

### Cek independen: kewajaran HAZ
Batas −2SD WHO pada beberapa umur (dihitung langsung dari tabel) dan distribusi HAZ.

In [ ]:
for sx, lab in [(1, "Laki-laki"), (2, "Perempuan")]:
    row = []
    for d_ in [0, 365, 730, 1095, 1460, 1826]:
        L, M, S = REF[(sx, d_)]; row.append(f"{d_//365} th: {M*(1-2*S):.1f} cm")
    print(lab, "| batas -2SD:", ", ".join(row))
plt.figure(figsize=(6, 3.2)); plt.hist(df["HAZ"], bins=120); plt.axvline(-2, color="r", ls="--")
plt.xlabel("HAZ"); plt.ylabel("Jumlah balita"); plt.title("Distribusi HAZ (garis merah = -2)"); plt.show()
print("Prevalensi resmi SKI 2023 (laporan nasional): 21,5% | notebook (tertimbang):", f"{flow['prevalence_weighted']*100:.1f}%")

## 7. Skenario fitur
- **B**: 119 determinan hulu (tanpa antropometri saat ini)
- **A1**: B + berat badan saat ini (proxy leakage)
- **A2**: B + berat, tinggi/panjang, LILA, lingkar perut (meniru literatur)
- **L0**: tinggi + umur + jenis kelamin saja (komponen rumus HAZ)

Dikeluarkan dari semua skenario: alasan PMT (I49, turunan status gizi), pemahaman ibu tentang stunting (G03.b dst.), posisi ukur, ID, penimbang, kode kabupaten, item >95% kosong.

In [ ]:
col = df.columns
def pick(prefixes): return [c for c in col if any(c.startswith(p) for p in prefixes)]
SENTINEL = {"I04.Usia kehamilan saat [NAMA] dilahirkan": [88],
            "I05.a.\tBerapa berat badan [NAMA] saat dilahirkan": [8888],
            "I07.Berapa panjang badan [NAMA] saat dilahirkan": [88, 88.8, 888],
            "H01.Berapa umur [NAMA] ketika pertama kali hamil?": [88, 98, 99]}
for c, vals in SENTINEL.items(): df.loc[df[c].isin(vals), c] = np.nan
BW, BL = "I05.a.\tBerapa berat badan [NAMA] saat dilahirkan", "I07.Berapa panjang badan [NAMA] saat dilahirkan"
df.loc[df[BW] > 6000, BW] = np.nan
df.loc[~df[BL].between(30, 65), BL] = np.nan
df["age_months"] = df["age_days"] / 30.4375

DET = (["1.Provinsi", "5. Klasifikasi Desa/Kelurahan", "4. Jenis Kelamin", "age_months", "8. Pendidikan tertinggi",
        "9. Status Pekerjaan", "11. Kepemilikan Jaminan Kesehatan", "3. Hubungan dengan KRT"]
       + pick(["A12.", "A22.", "B01.", "B07.a", "B16.", "G01", "G03.a", "G07.", "G11.", "G21.", "G32.", "H01.", "H09.",
               "H29.", "H32.", "I01.", "I04.", "I05.a", "I07.", "I09.", "I12", "I15", "I16.", "I37.", "I50", "150.", "J01.b"])
       + [c for c in col[128:166]])
DET = list(dict.fromkeys(DET))
ANTHRO = {"weight": "J01.c.Berat Badan (kg)", "height": HGT, "muac": "J07.b.Lingkar Lengan Atas (cm)", "waist": "J03.b.Lingkar Perut (Cm)"}
SETS = {"B (determinants only)": DET, "A1 (B + weight)": DET + [ANTHRO["weight"]],
        "A2 (B + all anthropometry)": DET + list(ANTHRO.values()), "L0 (height + age + sex)": [HGT, "age_months", "4. Jenis Kelamin"]}
for k, v in SETS.items(): print(f"{k:30s} {len(v)} variabel")
assert len(DET) == 119, "Jumlah determinan berbeda dari naskah (119) — cek urutan/nama kolom file Excel"
assert not any(c.startswith("I49") for c in DET) and HGT not in DET and ANTHRO["weight"] not in DET

## 8. Pembagian data berbasis rumah tangga
Saudara kandung dalam satu rumah tangga tidak boleh ada di data latih dan uji sekaligus.

In [ ]:
groups = df["ID RT"].values
folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(df, y, groups))
tr, te = folds[0]; yte = y[te]
overlap = len(set(groups[tr]) & set(groups[te]))
print("Rumah tangga yang muncul di latih & uji:", overlap)
assert overlap == 0
for k, v in {"train_n": len(tr), "test_n": len(te), "households_train": len(set(groups[tr])), "households_test": len(set(groups[te]))}.items():
    check(f"Split: {k}", v, EXPECTED["flow"][k], TOL_COUNT)
check("Split: test_prev", yte.mean(), EXPECTED["flow"]["test_prev"], TOL_PREV)

## 9. Model
Pra-pemrosesan berada **di dalam pipeline** (di-fit hanya pada data latih). Ketidakseimbangan kelas ditangani dengan `class_weight` (tanpa SMOTE).

In [ ]:
def make_pre(X, scale):
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]; cat = [c for c in X.columns if c not in num]
    nsteps = [("imp", SimpleImputer(strategy="median", add_indicator=True))] + ([("sc", StandardScaler())] if scale else [])
    return ColumnTransformer([("num", Pipeline(nsteps), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False))]), cat)])
def make_hgb(X):
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]; cat = [c for c in X.columns if c not in num]
    pre = ColumnTransformer([("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
              ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1))]), cat),
              ("num", "passthrough", num)], verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
        categorical_features=[True]*len(cat) + [False]*len(num), class_weight="balanced", early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, random_state=SEED))])
def models(X):
    return {"Logistic Regression": Pipeline([("pre", make_pre(X, True)), ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced"))]),
            "Random Forest": Pipeline([("pre", make_pre(X, False)), ("clf", RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                max_features="sqrt", max_samples=0.5, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1))]),
            "Gradient Boosting (HGB)": make_hgb(X)}
def metrics(yt, p, thr=0.5):
    yp = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
    return {"AUC-ROC": roc_auc_score(yt, p), "PR-AUC": average_precision_score(yt, p), "Accuracy": accuracy_score(yt, yp),
            "Recall": recall_score(yt, yp), "Specificity": tn/(tn+fp), "F1": f1_score(yt, yp), "MCC": matthews_corrcoef(yt, yp)}

## 10. Latih & evaluasi 12 kombinasi model–skenario

In [ ]:
rows, preds = [], {}
p0 = np.full(len(te), y[tr].mean()); r = metrics(yte, p0); r.update(scenario="Baseline", model="Majority/prior"); rows.append(r)
for s, feats in SETS.items():
    X = df[feats]
    for mname, pipe in models(X).items():
        t = time.time(); pipe.fit(X.iloc[tr], y[tr]); p = pipe.predict_proba(X.iloc[te])[:, 1]
        r = metrics(yte, p); r.update(scenario=s, model=mname); rows.append(r); preds[(s, mname)] = p
        print(f"{s:28s} {mname:24s} AUC={r['AUC-ROC']:.4f} Acc={r['Accuracy']:.4f}  ({time.time()-t:.0f} s)")
res = pd.DataFrame(rows)[["scenario", "model", "AUC-ROC", "PR-AUC", "Accuracy", "Recall", "Specificity", "F1", "MCC"]]
res.round(4)

### Cross-check metrik model vs naskah (Tabel 2)

In [ ]:
for _, r in res.iterrows():
    exp = EXPECTED["metrics"][f"{r.scenario}|{r.model}"]
    for k in ["AUC-ROC", "Accuracy", "F1"]:
        check(f"{r.scenario} | {r.model} | {k}", r[k], exp[k], TOL_AUC)

## 11. Selisih AUC berpasangan (bootstrap, Tabel 3)

In [ ]:
rng = np.random.default_rng(SEED); boot = [rng.integers(0, len(te), len(te)) for _ in range(N_BOOT)]
drows = []
for mname in ["Logistic Regression", "Random Forest", "Gradient Boosting (HGB)"]:
    pb = preds[("B (determinants only)", mname)]
    for sa in ["A1 (B + weight)", "A2 (B + all anthropometry)", "L0 (height + age + sex)"]:
        pa = preds[(sa, mname)]
        dd = [roc_auc_score(yte[b], pa[b]) - roc_auc_score(yte[b], pb[b]) for b in boot]
        obs = roc_auc_score(yte, pa) - roc_auc_score(yte, pb)
        drows.append({"model": mname, "comparison": f"{sa} vs B", "delta_AUC": obs, "CI_low": np.percentile(dd, 2.5),
                      "CI_high": np.percentile(dd, 97.5), "p_boot": np.mean(np.array(dd) <= 0)})
        check(f"ΔAUC {mname} | {sa} vs B", obs, EXPECTED["delta"][f"{mname}|{sa} vs B"], TOL_AUC)
pd.DataFrame(drows).round(4)

## 12. Validasi silang 5-fold berbasis rumah tangga (HGB) — opsional

In [ ]:
if RUN_CV:
    for s, feats in SETS.items():
        X = df[feats]; aucs = []
        for a, b in folds:
            pipe = make_hgb(X).fit(X.iloc[a], y[a]); aucs.append(roc_auc_score(y[b], pipe.predict_proba(X.iloc[b])[:, 1]))
        print(f"{s:28s} AUC = {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")
        check(f"CV5 HGB mean AUC | {s}", np.mean(aucs), EXPECTED["cv"][s], TOL_AUC)

## 13. Permutation importance (HGB) — opsional
Menunjukkan variabel apa yang sebenarnya dipakai model. Pada model bocor (A2), tinggi badan dan umur seharusnya mendominasi.

In [ ]:
if RUN_PERM:
    for s in ["A2 (B + all anthropometry)", "B (determinants only)"]:
        X = df[SETS[s]]; pipe = make_hgb(X).fit(X.iloc[tr], y[tr]); Xte = X.iloc[te]
        sub = np.random.default_rng(SEED).choice(len(Xte), size=min(8000, len(Xte)), replace=False)
        pi = permutation_importance(pipe, Xte.iloc[sub], yte[sub], scoring="roc_auc", n_repeats=3, random_state=SEED)
        top = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False).head(10)
        print(f"\n{s}\n", top.round(4).to_string())

## 14. Kurva ROC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, mname in zip(axes, ["Logistic Regression", "Random Forest", "Gradient Boosting (HGB)"]):
    for s in SETS:
        fpr, tpr, _ = roc_curve(yte, preds[(s, mname)]); ax.plot(fpr, tpr, label=f"{s.split(' (')[0]} (AUC={roc_auc_score(yte, preds[(s, mname)]):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_title(mname); ax.set_xlabel("False positive rate"); ax.legend(fontsize=8)
axes[0].set_ylabel("True positive rate"); plt.tight_layout(); plt.show()

## 15. Ringkasan cross-check
Jika semua ✅, hasil naskah terkonfirmasi. Jika ada ⚠️: (1) periksa versi scikit-learn (harus 1.8.0); (2) pastikan file Excel sama persis dengan yang dipakai (86.364 baris, 171 kolom); (3) perbedaan kecil (≤ 0,005) karena perbedaan platform/versi pustaka masih wajar dan tidak mengubah kesimpulan.

In [ ]:
summary = pd.DataFrame(checks)
print(f"Total cek: {len(summary)} | ✅ cocok: {(summary.status=='✅').sum()} | ⚠️ berbeda: {(summary.status=='⚠️').sum()}")
res.to_csv("colab_metrics.csv", index=False); summary.to_csv("colab_crosscheck_summary.csv", index=False)
summary

In [ ]:
try:
    from google.colab import files
    files.download("colab_crosscheck_summary.csv"); files.download("colab_metrics.csv")
except ImportError:
    print("File tersimpan: colab_crosscheck_summary.csv, colab_metrics.csv")